# Agent 2: Batch Risk Assessor

**Purpose**: Analyze enriched samples from Agent 1 across the full batch to:
- Detect patterns (e.g., 7/10 samples show urea contamination)
- Assess batch-level risk (not sample-level)
- Flag inconclusive samples for retest
- Determine if batch should be quarantined, rejected, or retested

**Input**: `enriched_samples.json` (from Agent 1)
**Output**: `batch_risk_summary.json` → Input for Agent 3 (Report Writer)

## Step 1: Setup & Imports

In [ ]:
import os
import sys
import json
import time
from typing import Any, Dict, List, Optional
from collections import Counter, defaultdict
from dataclasses import dataclass
import logging
from statistics import mean, stdev

# LangGraph imports
from langgraph.graph import StateGraph, START, END

# Add foodguard lib to path
sys.path.insert(0, '//food-guard-ai')
import foodguard_lib as fgl

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Imports successful")

## Step 2: Define Risk Assessment Thresholds

In [ ]:
# Thresholds for batch-level risk assessment
BATCH_RISK_THRESHOLDS = {
    'CRITICAL_CONTAMINANT_THRESHOLD': 0.30,  # >30% of batch with critical adulterant
    'HIGH_CONTAMINANT_THRESHOLD': 0.50,      # >50% of batch with high-risk adulterant
    'MEDIUM_CONTAMINANT_THRESHOLD': 0.70,    # >70% of batch with medium-risk adulterant
    'INCONCLUSIVE_THRESHOLD': 0.20,           # >20% inconclusive: retest needed
    'CONFIDENCE_THRESHOLD': 0.65,             # Avg confidence must be >= this
}

ADULTERANT_RISK_LEVELS = {
    'authentic': 'None',
    'water_diluted': 'Medium',
    'urea_added': 'High',
    'ammonium_sulfate': 'High',
    'oil_surfactant': 'High',
    'formalin_added': 'Critical',
    'spoiled': 'Critical',
    'inconclusive': 'Unknown',
}

print("✓ Risk thresholds defined")

## Step 3: Batch Analysis Functions

In [ ]:
def analyze_batch_patterns(enriched_samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Analyze patterns across all samples in the batch.
    Returns dict with adulterant frequencies, confidence stats, etc.
    """
    if not enriched_samples:
        return {}

    adulterants = [s['ground_truth'] for s in enriched_samples]
    confidences = [s['confidence'] for s in enriched_samples]
    risk_levels = [ADULTERANT_RISK_LEVELS.get(s['ground_truth'], 'Unknown') for s in enriched_samples]

    # Count adulterants
    adulterant_counts = Counter(adulterants)
    adulterant_percentages = {k: round(v / len(enriched_samples), 3) for k, v in adulterant_counts.items()}

    # Confidence stats
    avg_confidence = round(mean(confidences), 3)
    confidence_range = (round(min(confidences), 3), round(max(confidences), 3))

    # Risk level distribution
    risk_counts = Counter(risk_levels)

    # Find inconclusive samples
    inconclusive_samples = [s['sample_id'] for s in enriched_samples if s['ground_truth'] == 'inconclusive']
    low_confidence_samples = [s['sample_id'] for s in enriched_samples if s['confidence'] < 0.50]

    return {
        'total_samples': len(enriched_samples),
        'adulterant_distribution': adulterant_percentages,
        'adulterant_counts': dict(adulterant_counts),
        'confidence_avg': avg_confidence,
        'confidence_range': confidence_range,
        'risk_level_distribution': dict(risk_counts),
        'inconclusive_samples': inconclusive_samples,
        'inconclusive_count': len(inconclusive_samples),
        'inconclusive_percentage': round(len(inconclusive_samples) / len(enriched_samples), 3),
        'low_confidence_samples': low_confidence_samples,
        'low_confidence_count': len(low_confidence_samples),
    }

print("✓ Pattern analysis function defined")

## Step 4: Determine Batch Decision

In [ ]:
def determine_batch_decision(patterns: Dict[str, Any]) -> Dict[str, Any]:
    """
    Based on patterns, determine batch decision:
    'QUARANTINE', 'REJECT', 'RETEST', 'ACCEPT', 'CONDITIONAL_ACCEPT'
    """
    decision = 'ACCEPT'
    reasoning = []
    recommended_action = []

    adulterant_dist = patterns.get('adulterant_distribution', {})
    avg_confidence = patterns.get('confidence_avg', 0.0)
    inconclusive_pct = patterns.get('inconclusive_percentage', 0.0)
    risk_dist = patterns.get('risk_level_distribution', {})

    # Check for critical contaminants
    critical_adulterants = ['formalin_added', 'spoiled']
    critical_pct = sum(adulterant_dist.get(a, 0) for a in critical_adulterants)

    if critical_pct > BATCH_RISK_THRESHOLDS['CRITICAL_CONTAMINANT_THRESHOLD']:
        decision = 'REJECT'
        reasoning.append(f"{critical_pct:.0%} of batch shows critical contamination (formalin/spoilage)")
        recommended_action.append("Reject entire batch. Do not proceed to market.")

    # Check for high contaminants
    high_adulterants = ['urea_added', 'ammonium_sulfate', 'oil_surfactant']
    high_pct = sum(adulterant_dist.get(a, 0) for a in high_adulterants)

    if decision == 'ACCEPT' and high_pct > BATCH_RISK_THRESHOLDS['HIGH_CONTAMINANT_THRESHOLD']:
        decision = 'QUARANTINE'
        reasoning.append(f"{high_pct:.0%} of batch shows high-risk adulterants")
        recommended_action.append("Quarantine batch pending investigation.")
        recommended_action.append("Conduct confirmatory testing with NABL reference methods.")

    # Check for inconclusive samples requiring retest
    if inconclusive_pct > BATCH_RISK_THRESHOLDS['INCONCLUSIVE_THRESHOLD']:
        decision = 'RETEST'
        reasoning.append(f"{inconclusive_pct:.0%} of samples are inconclusive")
        recommended_action.append("Retest flagged samples with additional modalities.")

    # Check average confidence
    if avg_confidence < BATCH_RISK_THRESHOLDS['CONFIDENCE_THRESHOLD']:
        if decision == 'ACCEPT':
            decision = 'CONDITIONAL_ACCEPT'
        reasoning.append(f"Average confidence ({avg_confidence:.0%}) below threshold")
        recommended_action.append("Flag low-confidence samples for manual review.")

    # Check for medium risk
    medium_pct = adulterant_dist.get('water_diluted', 0)
    if decision == 'ACCEPT' and medium_pct > BATCH_RISK_THRESHOLDS['MEDIUM_CONTAMINANT_THRESHOLD']:
        decision = 'CONDITIONAL_ACCEPT'
        reasoning.append(f"{medium_pct:.0%} of batch shows water dilution (medium risk)")
        recommended_action.append("Accept batch with quality downgrade.")

    # All authentic
    if decision == 'ACCEPT' and adulterant_dist.get('authentic', 0) == 1.0:
        reasoning.append("All samples certified authentic")
        recommended_action.append("Release batch for distribution.")

    return {
        'decision': decision,
        'reasoning': reasoning,
        'recommended_actions': recommended_action,
    }

print("✓ Batch decision logic defined")

## Step 5: Identify Retest Candidates

In [ ]:
def identify_retest_candidates(enriched_samples: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Flag samples that should be retested.
    - Inconclusive samples
    - Low confidence samples
    - Ambiguous (top 2 candidates close)
    """
    retest_needed = []

    for sample in enriched_samples:
        reasons = []

        # Check if inconclusive
        if sample['ground_truth'] == 'inconclusive':
            reasons.append('inconclusive')

        # Check if low confidence
        if sample['confidence'] < 0.60:
            reasons.append('low_confidence')

        # Check if ambiguous (used differentiator)
        if 'DIFF-' in sample.get('differentiator_used', ''):
            reasons.append('ambiguous_needed_differentiator')

        if reasons:
            retest_needed.append({
                'sample_id': sample['sample_id'],
                'current_prediction': sample['ground_truth'],
                'confidence': sample['confidence'],
                'retest_reasons': reasons,
                'recommendation': 'Request additional modality data or higher resolution testing',
            })

    return {
        'retest_required': len(retest_needed) > 0,
        'samples_flagged': len(retest_needed),
        'samples': retest_needed,
    }

print("✓ Retest candidate identification defined")

## Step 6: LangGraph Node Function

In [ ]:
def batch_risk_assessor_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    Agent 2 node: Batch Risk Assessor.
    Analyzes enriched samples, identifies patterns, and determines batch-level risk.
    """
    start_time = time.time()

    enriched_samples = state.get('enriched_samples', [])
    batch_id = state.get('batch_id', 'unknown')

    logger.info(f"Batch Risk Assessor analyzing {len(enriched_samples)} samples")

    # Analyze patterns
    patterns = analyze_batch_patterns(enriched_samples)
    logger.info(f"Pattern analysis: {patterns}")

    # Determine batch decision
    decision_data = determine_batch_decision(patterns)
    logger.info(f"Batch decision: {decision_data['decision']}")

    # Identify retest candidates
    retest_data = identify_retest_candidates(enriched_samples)
    logger.info(f"Retest candidates: {retest_data['samples_flagged']}")

    # Execution time
    execution_time = (time.time() - start_time) * 1000  # ms
    exec_id = fgl.generate_id('EXEC')

    reasoning = f"Analyzed {len(enriched_samples)} enriched samples. " \
                f"Adulterant distribution: {patterns.get('adulterant_distribution', {})}. " \
                f"Decision: {decision_data['decision']}. " \
                f"Flagged {retest_data['samples_flagged']} samples for retest."

    # Build batch risk summary
    batch_risk_summary = {
        'batch_id': batch_id,
        'batch_patterns': patterns,
        'batch_decision': decision_data['decision'],
        'decision_reasoning': decision_data['reasoning'],
        'recommended_actions': decision_data['recommended_actions'],
        'retest_analysis': retest_data,
        'total_samples': len(enriched_samples),
    }

    # Return updated state
    state['batch_risk_summary'] = batch_risk_summary
    state['agent_execution_id'] = exec_id
    state['reasoning'] = reasoning
    state['execution_time_ms'] = execution_time

    logger.info(f"Batch Risk Assessor complete")

    return state

print("✓ LangGraph node function defined")

## Step 7: Build LangGraph

In [ ]:
# Build graph
builder = StateGraph(dict)
builder.add_node("batch_risk_assessor", batch_risk_assessor_node)
builder.set_entry_point("batch_risk_assessor")
builder.add_edge("batch_risk_assessor", END)

graph = builder.compile()

print("✓ LangGraph compiled")
print("\nGraph structure:")
print(graph.get_graph().draw_ascii())

## Step 8: Test with Sample Data from Agent 1

In [ ]:
# Create test enriched samples (simulating Agent 1 output)
test_enriched_samples = [
    {
        'sample_id': 'S001',
        'ground_truth': 'authentic',
        'confidence': 0.92,
        'risk_level': 'None',
        'status': 'Safe',
        'differentiator_used': 'none',
    },
    {
        'sample_id': 'S002',
        'ground_truth': 'authentic',
        'confidence': 0.88,
        'risk_level': 'None',
        'status': 'Safe',
        'differentiator_used': 'none',
    },
    {
        'sample_id': 'S003',
        'ground_truth': 'urea_added',
        'confidence': 0.75,
        'risk_level': 'High',
        'status': 'Unsafe',
        'differentiator_used': 'DIFF-1',
    },
    {
        'sample_id': 'S004',
        'ground_truth': 'inconclusive',
        'confidence': 0.35,
        'risk_level': 'Unknown',
        'status': 'Inconclusive',
        'differentiator_used': 'N/A',
    },
    {
        'sample_id': 'S005',
        'ground_truth': 'water_diluted',
        'confidence': 0.70,
        'risk_level': 'Medium',
        'status': 'Caution',
        'differentiator_used': 'DIFF-5',
    },
]

# Run graph
initial_state = {
    'enriched_samples': test_enriched_samples,
    'batch_id': fgl.generate_batch_id(),
}

output_state = graph.invoke(initial_state)

print(f"✓ Graph executed successfully")
print(f"Batch ID: {output_state['batch_id']}")
print(f"Batch Decision: {output_state['batch_risk_summary']['batch_decision']}")

## Step 9: Display Results

In [ ]:
# Pretty-print batch risk summary
summary = output_state['batch_risk_summary']

print("\n" + "="*80)
print("BATCH RISK SUMMARY")
print("="*80)
print(f"\nBatch ID: {summary['batch_id']}")
print(f"Total Samples: {summary['total_samples']}")
print(f"\nBatch Decision: {summary['batch_decision']}")

print("\nAdulterant Distribution:")
for adulterant, pct in summary['batch_patterns']['adulterant_distribution'].items():
    print(f"  {adulterant}: {pct:.0%}")

print(f"\nConfidence Stats:")
print(f"  Average: {summary['batch_patterns']['confidence_avg']:.0%}")
print(f"  Range: {summary['batch_patterns']['confidence_range']}")

print(f"\nDecision Reasoning:")
for reason in summary['decision_reasoning']:
    print(f"  - {reason}")

print(f"\nRecommended Actions:")
for action in summary['recommended_actions']:
    print(f"  → {action}")

print(f"\nRetest Analysis:")
retest = summary['retest_analysis']
print(f"  Retest Required: {retest['retest_required']}")
print(f"  Samples Flagged: {retest['samples_flagged']}")
for sample in retest['samples']:
    print(f"    - {sample['sample_id']}: {sample['retest_reasons']}")

## Step 10: Save Batch Risk Summary to JSON

In [ ]:
# Save batch risk summary
output_path = '/food-guard-ai/output/batch_risk_summary.json'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

output_json = {
    'batch_id': output_state['batch_id'],
    'agent': 'Batch_Risk_Assessor',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'batch_risk_summary': output_state['batch_risk_summary'],
}

with open(output_path, 'w') as f:
    json.dump(output_json, f, indent=2)

print(f"✓ Batch risk summary saved to: {output_path}")
print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")